# 02 — Integração dos grafos de arquitetura e estrutura

**Estudo:** Creche Pré-Escola Tipo 2, FNDE, revisão 2025–R03.

**Objetivo desta etapa:** repetir o procedimento de leitura IFC e conversão IFC–LBD em dois blocos — arquitetura e estrutura —, verificar as saídas separadamente e identificar as dificuldades que antecedem a integração semântica. O enriquecimento corresponde à ampliação dos objetos e propriedades disponíveis para consulta.

O primeiro teste estrutural utiliza **TIPO2-SMT-MOD_R03.ifc**, o menor dos dois arquivos do pacote estrutural oficial. O arquivo SCO fica para uma etapa posterior. As duas execuções usam o mesmo motor e os mesmos parâmetros, com diretórios e bases de URI distintos.

**Sequência:** preparar → importar e converter arquitetura → repetir para estrutura → verificar → analisar alertas → registrar as pendências. Não são criadas correspondências automáticas entre elementos dos modelos. Esta etapa não realiza detecção geométrica de interferências.

**Referências:** Bonduel et al. (2018), [artigo e método IFCtoLBD](https://ceur-ws.org/Vol-2159/04paper.pdf); [IFCtoLBD 2.49.0](https://github.com/jyrkioraskari/IFCtoLBD/releases/tag/2.49.0); [fonte oficial FNDE](https://www.gov.br/fnde/pt-br/acesso-a-informacao/acoes-e-programas/programas/proinfancia/projetos-arquitetonicos-para-construcao/projeto-tipo-2). As rotinas de execução e avaliação são adaptações do estudo, apoiadas na documentação técnica indicada em cada etapa.

Executar as células na ordem. Cada bloco preserva seu IFC, grafo, inventário, comparação, registro e log. Uma divergência é um resultado para investigação; não determina, isoladamente, um erro de projeto.

**Situação desta versão:** diagnóstico da tentativa de integração. As saídas das disciplinas permanecem separadas. As correspondências e os conflitos novos não são resolvidos automaticamente.


## 1. Preparação do ambiente

Instala as bibliotecas, importa as funções e cria uma pasta exclusiva para o notebook 2. As versões IFCtoLBD 2.49.0, IfcOpenShell 0.8.4, RDFLib 7.1.4 e pySHACL 0.30.1 mantêm continuidade com o primeiro procedimento.

Fontes: [IfcOpenShell — exemplos](https://docs.ifcopenshell.org/ifcopenshell-python/code_examples.html), [RDFLib — navegação](https://rdflib.readthedocs.io/en/latest/intro_to_graphs/), [pySHACL — uso como módulo](https://github.com/RDFLib/pySHACL#python-module-use).


In [1]:
import sys, subprocess, os, json, hashlib, zipfile, shutil, time, re
from pathlib import Path
from urllib.request import urlretrieve
from collections import Counter, defaultdict

if os.environ.get("CIV2155_AMBIENTE_PREPARADO") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "ifcopenshell==0.8.4", "rdflib==7.1.4", "pyshacl==0.30.1"], check=True)

import ifcopenshell
import ifcopenshell.util.element as elemento_ifc
import ifcopenshell.util.unit as unidade_ifc
import pandas as pd
from rdflib import Graph, Dataset, Namespace, URIRef, Literal, RDF, RDFS, XSD
from pyshacl import validate

ROOT = Path(os.environ.get("CIV2155_PASTA", "/content/notebook2" if Path("/content").is_dir() else "notebook2")).resolve()
PASTA_DADOS, PASTA_RESULTADOS = ROOT/"dados", ROOT/"resultados"
for pasta in [PASTA_DADOS, PASTA_RESULTADOS]: pasta.mkdir(parents=True, exist_ok=True)
BOT = Namespace("https://w3id.org/bot#")
PROPS = Namespace("http://lbd.arch.rwth-aachen.de/props#")
PROV = Namespace("http://www.w3.org/ns/prov#")
BASE = "https://github.com/prcgui/bim-ontology-CIV2155/"
def sha256(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def salvar_json(p, obj):
    Path(p).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
print("Pasta do notebook 2:", ROOT)


Pasta do notebook 2: C:\Users\prcgu\OneDrive\Documentos\ChatGPT\Artigo da Disciplina BIM\desenvolvimento\notebook2_execucao


## 2. Arquivos de entrada e integridade

Os ZIPs preservam os bytes dos IFCs extraídos dos pacotes oficiais do FNDE. A função confere os hashes do ZIP e do IFC, extraindo somente o arquivo esperado. Cada disciplina recebe seu diretório próprio.

Arquitetura: [pacote oficial](https://www.fnde.gov.br/digap/projetos/Creche_Pre_Escola_Tipo_2/R03_2025/IFC/Projeto_Arquitetonico.rar). Estrutura: [pacote oficial](https://www.fnde.gov.br/digap/projetos/Creche_Pre_Escola_Tipo_2/R03_2025/IFC/Projeto_Estrutural.rar). A seleção SMT é explícita; os resultados não representam todo o pacote estrutural.


In [2]:
ENTRADAS = {'arquitetura': {'disciplina': 'Arquitetura', 'arquivo': 'TIPO2-ARQ-DTV_R03.ifc', 'url': 'https://raw.githubusercontent.com/prcgui/bim-ontology-CIV2155/main/dados/fnde/arquitetura/TIPO2-ARQ-DTV_R03.ifc.zip', 'sha256_zip': '19053ddc0698742ab28b1af2b8ee573bfe96ef5fec4f148d115cc4c8d8bfc770', 'sha256_ifc': '0975bf1394694eff8746e5e49232bdf950c36c37c858edb715c9e7bc8f65603a'}, 'estrutura': {'disciplina': 'Estrutura (SMT)', 'arquivo': 'TIPO2-SMT-MOD_R03.ifc', 'url': 'https://raw.githubusercontent.com/prcgui/bim-ontology-CIV2155/main/dados/fnde/estrutura/TIPO2-SMT-MOD_R03.ifc.zip', 'sha256_zip': '438a18dba8b2b656383127d9fbcd0ebf44bd4e55a07bfd822af8026209982048', 'sha256_ifc': '2d94e5927884963b4bb92720cf7c79fdfd9b968ca1c5117e165fe88afbfd692a'}}

def importar_ifc(chave):
    entrada = ENTRADAS[chave]
    pasta = PASTA_DADOS/chave
    pasta.mkdir(exist_ok=True)
    pacote = pasta/(entrada["arquivo"]+".zip")
    if not pacote.exists(): urlretrieve(entrada["url"], pacote)
    if sha256(pacote) != entrada["sha256_zip"]:
        raise ValueError("ZIP diferente do documentado: "+chave)
    with zipfile.ZipFile(pacote) as z:
        conteudo = z.read(entrada["arquivo"])
    arquivo = pasta/entrada["arquivo"]
    arquivo.write_bytes(conteudo)
    if sha256(arquivo) != entrada["sha256_ifc"]:
        raise ValueError("IFC diferente do documentado: "+chave)
    modelo = ifcopenshell.open(arquivo)
    print(chave, arquivo.name, modelo.schema, arquivo.stat().st_size, "bytes")
    return arquivo, modelo


## 3. Motor de conversão e correção textual documentada

Reutiliza o IFCtoLBD 2.49.0 e as duas correções textuais já examinadas: códigos Unicode da tabela `StringOperations` e preservação dos apóstrofos no token de `RDFWriter`. A preparação e os dez testes são executados uma vez; as duas disciplinas usam as mesmas classes corrigidas.

Base técnica: [buildingSMART — IFC String Encoding](https://technical.buildingsmart.org/resources/ifcimplementationguidance/string-encoding/), [Oracle — Unicode Escapes, §3.3](https://docs.oracle.com/javase/specs/jls/se21/html/jls-3.html#jls-3.3) e [fontes e decisões da adaptação](https://github.com/prcgui/bim-ontology-CIV2155/tree/main/referencias#codifica%C3%A7%C3%A3o-de-caracteres-e-interven%C3%A7%C3%A3o-local). A implementação específica é local; não é uma correção publicada pelos autores.

No Colab, instala Java 21 se necessário. O pacote dos autores é conferido por SHA-256 antes da extração.


In [3]:
MOTOR = ROOT/"motor"
MOTOR.mkdir(exist_ok=True)
URL_MOTOR = "https://github.com/jyrkioraskari/IFCtoLBD/releases/download/2.49.0/IFCtoLBD.zip"
HASH_MOTOR = "49dc8833418cadf552fd688eca8f56499c5d1dad8783712aab56825b84bba14e"
pacote_motor = MOTOR/"IFCtoLBD.zip"
if not pacote_motor.exists(): urlretrieve(URL_MOTOR, pacote_motor)
if sha256(pacote_motor) != HASH_MOTOR: raise ValueError("Pacote do motor não confere.")
if not (MOTOR/"IFCtoLBD").is_dir():
    with zipfile.ZipFile(pacote_motor) as z: z.extractall(MOTOR)
java = Path(os.environ.get("CIV2155_JAVA", "/usr/lib/jvm/java-21-openjdk-amd64/bin/java"))
if not java.is_file():
    if os.name == "nt": raise RuntimeError("Definir CIV2155_JAVA com o caminho do Java 21 JDK.")
    subprocess.run(["apt-get", "-qq", "update"], check=True, stdout=subprocess.DEVNULL)
    subprocess.run(["apt-get", "-qq", "install", "-y", "openjdk-21-jdk-headless"], check=True, stdout=subprocess.DEVNULL)
classpath = str(MOTOR/"IFCtoLBD"/"IFCtoLBD-Desktop_Java_21_lib"/"*")
classe_cli = "org.linkedbuildingdata.ifc2lbd.IFCtoLBDConverter_CLI"
print(subprocess.run([str(java),"-version"],capture_output=True,text=True,check=True).stderr.splitlines()[0])


openjdk version "21.0.12.1" 2026-08-18 LTS


In [4]:
# Mesma preparação textual do notebook 1; diretórios adaptados ao notebook 2.
import difflib
import hashlib
import json
import os
import re
import subprocess
import time
from pathlib import Path
from urllib.request import urlretrieve
from rdflib import Graph

pasta_correcao = ROOT / "correcao_textual"
pasta_correcao.mkdir(exist_ok=True)
pasta_resultados_corrigidos = PASTA_RESULTADOS / "preparacao_motor"
pasta_resultados_corrigidos.mkdir(exist_ok=True)
commit_fonte = "ca7811b7132a843b85beee748d255f9b5b709e07"
base_fonte = "https://raw.githubusercontent.com/jyrkioraskari/IFCtoLBD/" + commit_fonte + "/"
fontes = {
    "StringOperations.java": (
        "IFCtoLBD/src/main/java/org/linkedbuildingdata/ifc2lbd/core/utils/StringOperations.java",
        "5f7e65b41d6ce9acd00e03c38070f533a6902575a3f201629cc3b56708dced13",
    ),
    "RDFWriter.java": (
        "IFCtoRDF/src/main/java/be/ugent/RDFWriter.java",
        "6924f236cb17fd6cf89bcf69e393a874a22f964be84d4c1e6a46c5cc01433621",
    ),
}
registro_fontes, diferencas = {}, []
for nome, (caminho, esperado) in fontes.items():
    original = pasta_correcao / ("original_" + nome)
    urlretrieve(base_fonte + caminho, original)
    dados = original.read_bytes()
    if hashlib.sha256(dados).hexdigest() != esperado:
        raise ValueError("Fonte diferente da versão documentada: " + nome)
    texto_original = dados.decode("utf-8")
    texto_corrigido = texto_original
    if nome == "StringOperations.java":
        padrao = r'(line = line\.replace\("\\\\X2\\\\([0-9A-Fa-f]{4})\\\\X0\\\\", ")[^"\n]*("\);)'
        texto_corrigido, quantidade = re.subn(
            padrao, lambda m: m[1] + "\\u" + m[2] + m[3], texto_original
        )
        if quantidade != 66:
            raise ValueError("A tabela Unicode tem uma estrutura não prevista.")
    else:
        inicio = "private static String filterExtras(String txt) {\n"
        acrescimo = (
            '\t\t// CIV2155: preserve the content of the parser string token.\n'
            '\t\tif (txt.startsWith("\'")) return txt.substring(1);\n'
        )
        if texto_original.count(inicio) != 1:
            raise ValueError("A função de leitura de texto não foi localizada.")
        texto_corrigido = texto_original.replace(inicio, inicio + acrescimo, 1)
    destino = pasta_correcao / nome
    destino.write_text(texto_corrigido, encoding="utf-8")
    registro_fontes[nome] = {
        "url": base_fonte + caminho, "sha256_original": esperado,
        "sha256_corrigido": hashlib.sha256(destino.read_bytes()).hexdigest(),
    }
    diferencas.extend(difflib.unified_diff(
        texto_original.splitlines(keepends=True),
        texto_corrigido.splitlines(keepends=True),
        fromfile="original/" + nome, tofile="corrigido/" + nome,
    ))
(pasta_resultados_corrigidos / "alteracoes.patch").write_text("".join(diferencas), encoding="utf-8")

# O compilador é necessário somente para preparar as duas classes modificadas.
javac = Path(str(java)).with_name("javac.exe" if os.name == "nt" else "javac")
if not javac.is_file():
    subprocess.run(["apt-get", "-qq", "update"], check=True, stdout=subprocess.DEVNULL)
    subprocess.run(["apt-get", "-qq", "install", "-y", "openjdk-21-jdk-headless"],
                   check=True, stdout=subprocess.DEVNULL)
classes_corrigidas = pasta_correcao / "classes"
classes_corrigidas.mkdir(exist_ok=True)
subprocess.run([
    str(javac), "-encoding", "UTF-8", "-cp", classpath, "-d", str(classes_corrigidas),
    str(pasta_correcao / "StringOperations.java"), str(pasta_correcao / "RDFWriter.java"),
], check=True, capture_output=True, text=True)
classpath_corrigido = str(classes_corrigidas) + os.pathsep + classpath
javap = Path(str(java)).with_name("javap.exe" if os.name == "nt" else "javap")
for classe in ["be.ugent.RDFWriter", "org.linkedbuildingdata.ifc2lbd.core.utils.StringOperations"]:
    assinaturas = [
        subprocess.run([str(javap), "-public", "-cp", cp, classe],
                       check=True, capture_output=True, text=True).stdout
        for cp in [classpath, classpath_corrigido]
    ]
    if assinaturas[0] != assinaturas[1]:
        raise ValueError("A classe corrigida tem interface incompatível: " + classe)

# Testes independentes: caracteres isolados, sequências combinadas e apóstrofos.
codigo_teste = r'''
import org.linkedbuildingdata.ifc2lbd.core.utils.StringOperations;
public class TesteTextosCIV2155 {
 public static void main(String[] args) throws Exception {
  String[][] unicode = {
   {"BER\\X2\\00C7\\X0\\ARIO", "BER\u00C7ARIO"},
   {"SOL\\X2\\00C1\\X0\\RIO", "SOL\u00C1RIO"},
   {"Servi\\X2\\00E7\\X0\\o", "Servi\u00E7o"},
   {"CIRCULA\\X2\\00C700C3\\X0\\O", "CIRCULA\u00C7\u00C3O"},
   {"COZINHA", "COZINHA"},
   {"CASTELO D'\\X2\\00C1\\X0\\GUA", "CASTELO D'\u00C1GUA"}
  };
  for (String[] par : unicode)
   if (!StringOperations.handleUnicode(par[0]).equals(par[1]))
    throw new AssertionError("Unicode: " + par[0]);
  java.lang.reflect.Method filtro =
   Class.forName("be.ugent.RDFWriter").getDeclaredMethod("filterExtras", String.class);
  filtro.setAccessible(true);
  String[][] tokens = {
   {"'CASTELO D'\u00C1GUA", "CASTELO D'\u00C1GUA"},
   {"'COZINHA", "COZINHA"}, {"'A=B", "A=B"}, {"'d'agua", "d'agua"}
  };
  for (String[] par : tokens)
   if (!filtro.invoke(null, par[0]).equals(par[1]))
    throw new AssertionError("Token: " + par[0]);
  System.out.println("10/10 testes de texto aprovados.");
 }
}
'''
teste_java = pasta_correcao / "TesteTextosCIV2155.java"
teste_java.write_text(codigo_teste, encoding="utf-8")
subprocess.run([str(javac), "-encoding", "UTF-8", "-cp", classpath_corrigido,
                "-d", str(classes_corrigidas), str(teste_java)], check=True, capture_output=True)
teste = subprocess.run([str(java), "-cp", classpath_corrigido, "TesteTextosCIV2155"],
                       check=True, capture_output=True, text=True)
(pasta_resultados_corrigidos / "testes_textos.txt").write_text(teste.stdout, encoding="utf-8")
print(teste.stdout.strip())


salvar_json(pasta_resultados_corrigidos/"fontes_correcao.json", registro_fontes)


10/10 testes de texto aprovados.


## 4. Mesmo procedimento para as duas disciplinas

A função de inventário lê identificadores, classes, nomes, categorias de ambientes, pavimentos e unidades diretamente do IFC. A função de conversão executa o mesmo motor para cada arquivo e registra o comando e os hashes. As verificações comparam dados pelo `GlobalId`, preservando os textos originais.

**Ampliação explícita do escopo:** a opção `--hasIfc_based_elements=true` habilita a inclusão de elementos no percurso de conversão desta versão. Na [CLI dos autores](https://github.com/jyrkioraskari/IFCtoLBD/blob/ca7811b7132a843b85beee748d255f9b5b709e07/IFCtoLBD/src/main/java/org/linkedbuildingdata/ifc2lbd/IFCtoLBDConverter_CLI.java) ela é encaminhada a `setHasNonLBDElement`; o [código central](https://github.com/jyrkioraskari/IFCtoLBD/blob/ca7811b7132a843b85beee748d255f9b5b709e07/IFCtoLBD/src/main/java/org/linkedbuildingdata/ifc2lbd/core/IFCtoLBDConverterCore.java#L1151-L1162) controla a inclusão. O notebook 2 examina os elementos além dos ambientes, mantendo registro dessa diferença de configuração.

Fontes: [IfcOpenShell — get_container e get_pset](https://docs.ifcopenshell.org/autoapi/ifcopenshell/util/element/index.html), [calculate_unit_scale](https://docs.ifcopenshell.org/autoapi/ifcopenshell/util/unit/index.html#ifcopenshell.util.unit.calculate_unit_scale) e [RDFLib — subjects, objects e value](https://rdflib.readthedocs.io/en/latest/intro_to_graphs/). A unidade é identificada no IFC; não é presumida a partir do nome do arquivo.


In [5]:
def inventariar(modelo, chave):
    escala = unidade_ifc.calculate_unit_scale(modelo)
    linhas = []
    for e in modelo.by_type("IfcProduct"):
        pavimento = elemento_ifc.get_container(e, ifc_class="IfcBuildingStorey")
        linhas.append({"modelo": chave, "classe_ifc": e.is_a(), "GlobalId": e.GlobalId,
            "Name": e.Name, "LongName": getattr(e,"LongName",None),
            "Category": elemento_ifc.get_pset(e,"Pset_SpaceCommon","Category",should_inherit=False) if e.is_a("IfcSpace") else None,
            "pavimento_ifc": None if pavimento is None else pavimento.GlobalId})
    tabela = pd.DataFrame(linhas)
    pasta = PASTA_RESULTADOS/chave; pasta.mkdir(exist_ok=True)
    tabela.to_csv(pasta/"inventario_ifc.csv",index=False,encoding="utf-8-sig")
    pavimentos = pd.DataFrame([{"modelo":chave,"GlobalId":e.GlobalId,"nome":e.Name,
        "cota_declarada":e.Elevation,"escala_para_metro":escala,
        "cota_em_metros":None if e.Elevation is None else e.Elevation*escala}
        for e in modelo.by_type("IfcBuildingStorey")])
    pavimentos.to_csv(pasta/"pavimentos_ifc.csv",index=False,encoding="utf-8-sig")
    print(tabela.groupby("classe_ifc").size().to_string())
    print("Fator de conversão da unidade de comprimento para metro:",escala)
    return tabela, pavimentos, escala

def converter(arquivo, chave):
    pasta = PASTA_RESULTADOS/chave; pasta.mkdir(exist_ok=True)
    destino = pasta/"grafo.ttl"
    comando = [str(java),"-Xmx4g","-Dfile.encoding=UTF-8","-cp",classpath_corrigido,
        classe_cli,str(arquivo),"--url="+BASE+"id/notebook2/"+chave+"/",
        "--target_file="+str(destino),"--level=1","--hasBuildingElements=true",
        "--hasBuildingElementProperties=true","--hasIfc_based_elements=true",
        "--hasSeparateBuildingElementsModel=false","--hasSeparatePropertiesModel=false",
        "--propertiesAsPropertySets=false","--hasUnits=true","--hasGeometry=false","--ifcOWL=false"]
    inicio=time.perf_counter()
    hash_entrada=sha256(arquivo)
    print("Convertendo",chave,"...")
    with (pasta/"conversao.log").open("w",encoding="utf-8") as log:
        processo=subprocess.run(comando,cwd=pasta,stdout=log,stderr=subprocess.STDOUT,timeout=1800)
    processo.check_returncode()
    if sha256(arquivo)!=hash_entrada:raise ValueError("IFC de entrada alterado.")
    grafo=Graph().parse(destino,format="turtle")
    registro={"modelo":chave,"motor":"IFCtoLBD","versao":"2.49.0","sha256_motor":HASH_MOTOR,
        "sha256_ifc":hash_entrada,"sha256_grafo":sha256(destino),"comando":comando,
        "triplas":len(grafo),"duracao_segundos":round(time.perf_counter()-inicio,1)}
    salvar_json(pasta/"registro_conversao.json",registro)
    print(chave,len(grafo),"triplas em",registro["duracao_segundos"],"s")
    return grafo

def indice_guid(grafo):
    por_guid=defaultdict(set)
    for recurso,guid in grafo.subject_objects(PROPS.globalIdIfcRoot_attribute_simple):
        por_guid[str(guid)].add(recurso)
    return por_guid

def pavimentos_no_grafo(grafo,recurso):
    # Sobe pelas relações de contenção e decomposição até os pavimentos.
    fila=[recurso];visitados=set();resultado=set()
    while fila:
        atual=fila.pop()
        if atual in visitados:continue
        visitados.add(atual)
        if (atual,RDF.type,BOT.Storey) in grafo:
            resultado.update(str(v) for v in grafo.objects(atual,PROPS.globalIdIfcRoot_attribute_simple))
        for pred in [BOT.containsElement,BOT.hasSpace,BOT.hasSubElement,BOT.hasStorey]:
            fila.extend(grafo.subjects(pred,atual))
    return resultado

def avaliar(modelo,grafo,inventario,chave):
    indice=indice_guid(grafo);linhas=[];relacoes=[]
    for e in modelo.by_type("IfcProduct"):
        recursos=indice.get(e.GlobalId,set())
        resultado_id="preservado" if len(recursos)==1 else ("ausente" if not recursos else "ambiguo")
        linhas.append({"GlobalId":e.GlobalId,"classe_ifc":e.is_a(),"campo":"GlobalId","IFC":e.GlobalId,
            "LBD":e.GlobalId if len(recursos)==1 else None,"resultado":resultado_id})
        if len(recursos)!=1:continue
        recurso=next(iter(recursos))
        campos=[("Name",e.Name,RDFS.label),("LongName",getattr(e,"LongName",None),PROPS.longNameIfcSpatialElement_attribute_simple)]
        if e.is_a("IfcSpace"):
            campos.append(("Category",elemento_ifc.get_pset(e,"Pset_SpaceCommon","Category",should_inherit=False),PROPS.category_property_simple))
        for campo,origem,pred in campos:
            if origem is None:continue
            valores=list(grafo.objects(recurso,pred))
            convertido=str(valores[0]) if len(valores)==1 else None
            linhas.append({"GlobalId":e.GlobalId,"classe_ifc":e.is_a(),"campo":campo,"IFC":origem,"LBD":convertido,
                "resultado":"preservado" if len(valores)==1 and origem==convertido else "divergente"})
        pavimento=elemento_ifc.get_container(e,ifc_class="IfcBuildingStorey")
        if pavimento is not None:
            obtidos=pavimentos_no_grafo(grafo,recurso)
            relacoes.append({"GlobalId":e.GlobalId,"pavimento_ifc":pavimento.GlobalId,"pavimentos_lbd":";".join(sorted(obtidos)),
                "resultado":"preservado" if obtidos=={pavimento.GlobalId} else "divergente"})
    tabela=pd.DataFrame(linhas);relacoes=pd.DataFrame(relacoes,columns=["GlobalId","pavimento_ifc","pavimentos_lbd","resultado"])
    pasta=PASTA_RESULTADOS/chave
    tabela.to_csv(pasta/"comparacao_campos.csv",index=False,encoding="utf-8-sig")
    relacoes.to_csv(pasta/"comparacao_pavimentos.csv",index=False,encoding="utf-8-sig")
    tabela.groupby(["campo","resultado"]).size().rename("quantidade").to_csv(pasta/"resumo_comparacao.csv",encoding="utf-8-sig")
    print(tabela.groupby(["campo","resultado"]).size().to_string())
    print("Relações com pavimentos:",relacoes["resultado"].value_counts().to_dict())
    print("GlobalId repetidos no IFC:",int(inventario["GlobalId"].duplicated(keep=False).sum()))
    return tabela,relacoes


## 5. Bloco A — Arquitetura

Importa o IFC, apresenta seu inventário, converte e compara os dados. Este bloco repete as mesmas funções do outro bloco, com variáveis, arquivos e URIs próprios.

Os resultados são observados; ausências ou divergências não são corrigidas pelo preenchimento de valores esperados. As fontes das operações estão nas etapas 2 a 4.


In [6]:
# Execução independente: arquitetura.
arquivo_arquitetura, modelo_arquitetura = importar_ifc("arquitetura")
inventario_arquitetura, pavimentos_arquitetura, escala_arquitetura = inventariar(modelo_arquitetura,"arquitetura")
grafo_arquitetura = converter(arquivo_arquitetura,"arquitetura")
comparacao_arquitetura, relacoes_arquitetura = avaliar(modelo_arquitetura,grafo_arquitetura,inventario_arquitetura,"arquitetura")


arquitetura TIPO2-ARQ-DTV_R03.ifc IFC4 27852645 bytes
classe_ifc
IfcBuilding              1
IfcBuildingStorey        2
IfcCovering            631
IfcDoor                 53
IfcFurniture           170
IfcLightFixture        130
IfcOpeningElement      184
IfcRailing              15
IfcRamp                  5
IfcRoof                152
IfcSanitaryTerminal    154
IfcShadingDevice        44
IfcSite                  1
IfcSpace                41
IfcStair                 1
IfcTank                  2
IfcWall                457
IfcWasteTerminal        34
IfcWindow               39
Fator de conversão da unidade de comprimento para metro: 1
Convertendo arquitetura ...
arquitetura 56961 triplas em 216.5 s
campo     resultado 
Category  preservado      41
GlobalId  preservado    2116
LongName  preservado      41
Name      preservado    2110
Relações com pavimentos: {'preservado': 1887, 'divergente': 184}
GlobalId repetidos no IFC: 0


## 6. Bloco B — Estrutura (SMT)

Importa o IFC, apresenta seu inventário, converte e compara os dados. Este bloco repete as mesmas funções do outro bloco, com variáveis, arquivos e URIs próprios.

Os resultados são observados; ausências ou divergências não são corrigidas pelo preenchimento de valores esperados. As fontes das operações estão nas etapas 2 a 4.


In [7]:
# Execução independente: estrutura.
arquivo_estrutura, modelo_estrutura = importar_ifc("estrutura")
inventario_estrutura, pavimentos_estrutura, escala_estrutura = inventariar(modelo_estrutura,"estrutura")
grafo_estrutura = converter(arquivo_estrutura,"estrutura")
comparacao_estrutura, relacoes_estrutura = avaliar(modelo_estrutura,grafo_estrutura,inventario_estrutura,"estrutura")


estrutura TIPO2-SMT-MOD_R03.ifc IFC4 2642306 bytes
classe_ifc
IfcBeam               2518
IfcBuilding              1
IfcBuildingStorey        1
IfcElementAssembly    1365
IfcPlate              1091
IfcRoof                252
IfcSite                  1
IfcVoidingFeature       27
Fator de conversão da unidade de comprimento para metro: 0.001
Convertendo estrutura ...
estrutura 84933 triplas em 45.0 s
campo     resultado 
GlobalId  preservado    5256
Name      preservado    5256
Relações com pavimentos: {'divergente': 5253}
GlobalId repetidos no IFC: 0


## 7. Alertas de integração

Compara a cobertura de objetos, os GUIDs e as cotas declaradas dos pavimentos, convertidas para metros. Pavimentos com nomes iguais, após desconsiderar apenas maiúsculas e minúsculas, são **candidatos de comparação**, não entidades equivalentes. Diferenças de cotas exigem examinar referenciais e posicionamentos; esta rotina não calcula coordenadas globais nem demonstra interferências físicas.

Fontes: [IfcOpenShell — unidades](https://docs.ifcopenshell.org/autoapi/ifcopenshell/util/unit/index.html), [RDFLib — navegação](https://rdflib.readthedocs.io/en/latest/intro_to_graphs/) e [Python — str.casefold](https://docs.python.org/3/library/stdtypes.html#str.casefold). Os critérios de alerta são definidos neste estudo e não são transcritos de um artigo.


In [8]:
resumo=[]
for chave,m,g,t,c in [("arquitetura",modelo_arquitetura,grafo_arquitetura,inventario_arquitetura,comparacao_arquitetura),
                       ("estrutura",modelo_estrutura,grafo_estrutura,inventario_estrutura,comparacao_estrutura)]:
    resumo.append({"modelo":chave,"produtos_ifc":len(t),"elementos_ifc":len(m.by_type("IfcElement")),
        "ambientes_ifc":len(m.by_type("IfcSpace")),"elementos_bot":len(set(g.subjects(RDF.type,BOT.Element))),
        "ambientes_bot":len(set(g.subjects(RDF.type,BOT.Space))),"triplas":len(g),
        "guid_ausentes":int(((c.campo=="GlobalId")&(c.resultado=="ausente")).sum()),
        "textos_divergentes":int(((c.campo!="GlobalId")&(c.resultado=="divergente")).sum())})
resumo_modelos=pd.DataFrame(resumo)
print(resumo_modelos.to_string(index=False))
resumo_modelos.to_csv(PASTA_RESULTADOS/"resumo_modelos.csv",index=False,encoding="utf-8-sig")

guids_comuns=set(inventario_arquitetura.GlobalId)&set(inventario_estrutura.GlobalId)
linhas=[]
for guid in sorted(guids_comuns):
    for chave,tabela in [("arquitetura",inventario_arquitetura),("estrutura",inventario_estrutura)]:
        for linha in tabela[tabela.GlobalId==guid].to_dict("records"):linhas.append(linha)
pd.DataFrame(linhas,columns=inventario_arquitetura.columns).to_csv(PASTA_RESULTADOS/"guid_compartilhados.csv",index=False,encoding="utf-8-sig")
print("GUIDs presentes nos dois IFCs:",len(guids_comuns),"— candidatos à inspeção, sem fusão automática.")

cotas=[]
for a in pavimentos_arquitetura.to_dict("records"):
    for b in pavimentos_estrutura.to_dict("records"):
        if a["nome"] and b["nome"] and a["nome"].strip().casefold()==b["nome"].strip().casefold():
            x,y=a["cota_em_metros"],b["cota_em_metros"]
            cotas.append({"nome":a["nome"],"guid_arquitetura":a["GlobalId"],"guid_estrutura":b["GlobalId"],
                "cota_arq_m":x,"cota_est_m":y,"diferenca_declarada_m":None if x is None or y is None else x-y,
                "interpretacao":"Verificar referenciais e posicionamentos; nome igual não comprova equivalência."})
comparacao_cotas=pd.DataFrame(cotas,columns=["nome","guid_arquitetura","guid_estrutura","cota_arq_m","cota_est_m","diferenca_declarada_m","interpretacao"])
comparacao_cotas.to_csv(PASTA_RESULTADOS/"pavimentos_candidatos.csv",index=False,encoding="utf-8-sig")
print(comparacao_cotas.to_string(index=False))
print("IfcSpace ausente no IFC estrutural: condição da entrada, não perda automática da conversão.")


     modelo  produtos_ifc  elementos_ifc  ambientes_ifc  elementos_bot  ambientes_bot  triplas  guid_ausentes  textos_divergentes
arquitetura          2116           2071             41           2071             41    56961              0                   0
  estrutura          5256           5253              0           5253              0    84933              0                   0
GUIDs presentes nos dois IFCs: 0 — candidatos à inspeção, sem fusão automática.
     nome       guid_arquitetura         guid_estrutura  cota_arq_m  cota_est_m  diferenca_declarada_m                                                                   interpretacao
COBERTURA 10xiyxAX_aIfEvvB7TXxdK 1hdQblJHj7gPb3fMM9XPsr        3.26         0.0                   3.26 Verificar referenciais e posicionamentos; nome igual não comprova equivalência.
IfcSpace ausente no IFC estrutural: condição da entrada, não perda automática da conversão.


## 8. Verificação SHACL

As regras exigem um GUID textual único por recurso identificado e um vínculo único de cada `bot:Space` com um `bot:Storey`. A segunda regra não exige espaços em um modelo estrutural que não os possui. A cobertura IFC–LBD é examinada separadamente; aprovação SHACL não demonstra conversão completa.

Fundamentação: Jäkel et al. (2025), [seção 5.2](https://doi.org/10.3390/buildings15050720), para a estratégia de consultas e SHACL; [recomendação SHACL do W3C](https://www.w3.org/TR/shacl/) e [pySHACL](https://github.com/RDFLib/pySHACL#python-module-use), para as interfaces. As regras abaixo são específicas deste estudo.


In [9]:
regras=Graph().parse(data="""
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix bot: <https://w3id.org/bot#> .
@prefix props: <http://lbd.arch.rwth-aachen.de/props#> .
@prefix ex: <https://github.com/prcgui/bim-ontology-CIV2155/shapes/notebook2/> .
ex:Identificador a sh:NodeShape ; sh:targetSubjectsOf props:globalIdIfcRoot_attribute_simple ;
 sh:property [ sh:path props:globalIdIfcRoot_attribute_simple ; sh:minCount 1 ; sh:maxCount 1 ; sh:datatype xsd:string ] .
ex:Ambiente a sh:NodeShape ; sh:targetClass bot:Space ;
 sh:property [ sh:path [sh:inversePath bot:hasSpace] ; sh:minCount 1 ; sh:maxCount 1 ; sh:class bot:Storey ] .
""",format="turtle")
regras.serialize(PASTA_RESULTADOS/"regras_notebook2.ttl",format="turtle")
validacoes=[]
for chave,g in [("arquitetura",grafo_arquitetura),("estrutura",grafo_estrutura)]:
    conforme,relatorio,texto=validate(g,shacl_graph=regras,inference="none",meta_shacl=True,inplace=False)
    relatorio.serialize(PASTA_RESULTADOS/chave/"relatorio_shacl.ttl",format="turtle")
    (PASTA_RESULTADOS/chave/"relatorio_shacl.txt").write_text(texto,encoding="utf-8")
    validacoes.append({"modelo":chave,"conforme_shacl":bool(conforme),"recursos_com_guid":len(set(g.subjects(PROPS.globalIdIfcRoot_attribute_simple,None))),
        "alvos_ambiente":len(set(g.subjects(RDF.type,BOT.Space)))})
salvar_json(PASTA_RESULTADOS/"validacao_shacl.json",validacoes)
print(pd.DataFrame(validacoes).to_string(index=False))


     modelo  conforme_shacl  recursos_com_guid  alvos_ambiente
arquitetura            True               2116              41
  estrutura            True               5256               0


## 9. Ponto de parada — problemas e decisões de integração pendentes

Esta célula registra os resultados observados e encerra o procedimento. Não altera os IFCs ou grafos, não funde recursos, não cria `owl:sameAs` e não corrige os vínculos divergentes. A inspeção de unidades e cotas é diagnóstica; os referenciais geométricos não são reconciliados.

Ausências na fonte, divergências na recuperação das relações e pendências de identidade são registradas separadamente. Uma relação que não é recuperada pelo critério implementado exige análise do mapeamento e da consulta antes de ser declarada perda do conversor. As diferenças de cota não demonstram, isoladamente, erro de projeto ou interferência física.

Fundamentação dos limites: [BOT — relações espaciais](https://w3c-lbd-cg.github.io/bot/), [W3C — RDF Datasets](https://www.w3.org/TR/rdf11-concepts/#section-dataset) e as fontes de conversão e consulta das etapas anteriores. Os critérios e o registro de pendências constituem uma adaptação deste estudo.


In [10]:
pendencias=[]
for chave,relacoes,inventario,comparacao in [
    ("arquitetura",relacoes_arquitetura,inventario_arquitetura,comparacao_arquitetura),
    ("estrutura",relacoes_estrutura,inventario_estrutura,comparacao_estrutura)]:
    divergentes=relacoes[relacoes.resultado=="divergente"].merge(inventario[["GlobalId","classe_ifc"]],on="GlobalId",how="left")
    divergentes.to_csv(PASTA_RESULTADOS/chave/"vinculos_pendentes.csv",index=False,encoding="utf-8-sig")
    for classe,grupo in divergentes.groupby("classe_ifc"):
        pendencias.append({"natureza":"divergência no critério de recuperação","modelo":chave,"quantidade":len(grupo),
            "observacao":"Vínculo com pavimento não recuperado como no IFC: "+classe,
            "decisao_pendente":"Examinar o mapeamento e o caminho de consulta; nenhuma correção aplicada."})
    for campo in ["GlobalId","Name","LongName","Category"]:
        qtd=int(((comparacao.campo==campo)&(comparacao.resultado!="preservado")).sum())
        if qtd:
            pendencias.append({"natureza":"cobertura ou valor divergente","modelo":chave,"quantidade":qtd,
                "observacao":"Registros não preservados pelo critério: "+campo,
                "decisao_pendente":"Inspecionar origem e saída; não preencher valores esperados."})
if not guids_comuns:
    pendencias.append({"natureza":"identidade entre modelos não estabelecida","modelo":"integração","quantidade":0,
        "observacao":"Nenhum GlobalId de IfcProduct compartilhado entre os dois arquivos.",
        "decisao_pendente":"Definir e validar critérios de correspondência. Não associar recursos apenas pelo nome."})
if escala_arquitetura!=escala_estrutura:
    pendencias.append({"natureza":"heterogeneidade de entrada","modelo":"integração","quantidade":2,
        "observacao":f"Fatores de comprimento para metro: arquitetura={escala_arquitetura}; estrutura={escala_estrutura}.",
        "decisao_pendente":"Considerar unidades e referenciais em uma futura verificação de alinhamento; não é erro por si só."})
for linha in comparacao_cotas.to_dict("records"):
    if linha["diferenca_declarada_m"] is not None and abs(linha["diferenca_declarada_m"])>1e-9:
        pendencias.append({"natureza":"referencial a verificar","modelo":"integração","quantidade":1,
            "observacao":f"Pavimentos chamados {linha['nome']}: cotas declaradas {linha['cota_arq_m']} m e {linha['cota_est_m']} m.",
            "decisao_pendente":"Não inferir equivalência. Examinar posicionamentos e referenciais em etapa posterior."})
if len(modelo_estrutura.by_type("IfcSpace"))==0:
    pendencias.append({"natureza":"condição da fonte","modelo":"estrutura","quantidade":0,
        "observacao":"O IFC SMT não contém IfcSpace.",
        "decisao_pendente":"Não atribuir automaticamente a ausência de ambientes a uma perda na conversão."})
quadro_pendencias=pd.DataFrame(pendencias)
quadro_pendencias.to_csv(PASTA_RESULTADOS/"problemas_e_pendencias.csv",index=False,encoding="utf-8-sig")
print(quadro_pendencias.to_string(index=False))
salvar_json(PASTA_RESULTADOS/"manifesto_estudo.json",{"entradas":ENTRADAS,
    "motor":"IFCtoLBD 2.49.0 com adaptação textual previamente documentada","sha256_motor":HASH_MOTOR,
    "escopo_estrutural":"SMT; SCO não incluído nesta etapa",
    "status":"Diagnóstico registrado; correspondências e conflitos sem solução aplicada.",
    "integracao_semantica":"Pendente. Grafos mantidos separadamente; nenhuma equivalência inferida.",
    "referenciais_geometricos":"Não reconciliados; não foi realizada detecção de interferências.",
    "escala_comprimento_para_metro":{"arquitetura":escala_arquitetura,"estrutura":escala_estrutura},
    "resumo":resumo_modelos.to_dict("records"),"shacl":validacoes,"pendencias":pendencias})
# Exporta somente as evidências deste diagnóstico, sem os arquivos de união.
pacote_resultados=ROOT/"02_diagnostico_arquitetura_estrutura.zip"
with zipfile.ZipFile(pacote_resultados,"w",zipfile.ZIP_DEFLATED) as z:
    for arquivo in sorted(PASTA_RESULTADOS.rglob("*")):
        if arquivo.is_file() and arquivo.name not in ["arquitetura_estrutura.trig","arquitetura_estrutura_uniao.ttl"]:
            z.write(arquivo,arquivo.relative_to(PASTA_RESULTADOS))
print("PONTO DE PARADA: análise encerrada com as pendências acima, sem aplicar soluções de integração.")
print("Pacote de evidências:",pacote_resultados)


                                 natureza      modelo  quantidade                                                           observacao                                                                                   decisao_pendente
   divergência no critério de recuperação arquitetura         184  Vínculo com pavimento não recuperado como no IFC: IfcOpeningElement                          Examinar o mapeamento e o caminho de consulta; nenhuma correção aplicada.
   divergência no critério de recuperação   estrutura        2518            Vínculo com pavimento não recuperado como no IFC: IfcBeam                          Examinar o mapeamento e o caminho de consulta; nenhuma correção aplicada.
   divergência no critério de recuperação   estrutura        1365 Vínculo com pavimento não recuperado como no IFC: IfcElementAssembly                          Examinar o mapeamento e o caminho de consulta; nenhuma correção aplicada.
   divergência no critério de recuperação   estrutura        109

## Continuidade da pesquisa

O resultado desta etapa é um diagnóstico reproduzível. A coexistência de informações de duas disciplinas não comprova equivalência entre objetos ou coordenação geométrica. A integração depende de decisões sobre identidade, organização espacial e interpretação das relações não recuperadas pelo critério adotado.

A discussão das pendências delimita o próximo passo metodológico. Esta versão encerra-se no registro, sem propor ou executar reparações. Os números devem ser interpretados junto com os inventários e as comparações exportadas.
